# Chapter 4 - Representing Data and Feature Engineering

This notebook contains code reproductions, theoretical explanations, and discussions based on Chapter 4 of *Introduction to Machine Learning with Python: A Guide for Data Scientists* by Andreas C. Müller and Sarah Guido.

## Chapter Summary

Feature engineering is the process of transforming raw data into meaningful features that can improve machine learning performance.

This chapter introduces techniques for handling categorical variables, creating new features, discretizing continuous variables, generating interaction terms, and selecting useful features from datasets.

Effective feature engineering can often improve model performance more than changing algorithms.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mglearn

from sklearn.model_selection import train_test_split

%matplotlib inline

# Categorical Variables

Many real-world datasets contain categorical features such as gender, country, product category, or payment method.

Machine learning algorithms usually require numerical input, so categorical variables must be converted into a numerical representation before training.

In [2]:
data = {
    "City": ["Jakarta", "Bandung", "Surabaya", "Jakarta"],
    "Rooms": [3, 2, 4, 3],
    "Price": [500, 350, 700, 550]
}

demo_df = pd.DataFrame(data)

demo_df

,City,Rooms,Price
0,Jakarta,3,500
1,Bandung,2,350
2,Surabaya,4,700
3,Jakarta,3,550


### Discussion

The dataset contains both numerical and categorical information.

The City column is categorical and cannot be directly used by most machine learning algorithms.

# One-Hot Encoding

One-Hot Encoding converts categorical variables into binary features.

Each category becomes a separate column, allowing machine learning models to process categorical information numerically.

In [3]:
pd.get_dummies(demo_df)

,Rooms,Price,City_Bandung,City_Jakarta,City_Surabaya
0,3,500,False,True,False
1,2,350,True,False,False
2,4,700,False,False,True
3,3,550,False,True,False


### Discussion

After applying one-hot encoding, each city is represented as a separate binary feature.

This prevents algorithms from incorrectly assuming an ordinal relationship between categories.

# Binning and Discretization

Binning transforms continuous numerical variables into discrete intervals.

This technique can simplify relationships within data and sometimes improve model performance.

In [4]:
from sklearn.datasets import make_blobs

X, y = make_blobs(
    n_samples=12,
    random_state=0
)

print(X[:, 0])

[ 3.54934659  1.9263585   0.0058752   1.12031365  1.7373078   2.36833522
 -0.49772229 -1.4811455   0.87305123 -0.66246781  0.74285061  2.49913075]


In [5]:
bins = np.linspace(
    -10,
    10,
    11
)

which_bin = np.digitize(
    X[:, 0],
    bins=bins
)

print(which_bin)

[7 6 6 6 6 7 5 5 6 5 6 7]


### Discussion

Discretization replaces continuous values with interval labels.

This can help linear models capture patterns that are not purely linear.

# Interaction Features

Interaction features are created by combining existing features.

These features allow models to capture relationships between variables that may not be visible when features are used independently.

In [6]:
from sklearn.preprocessing import PolynomialFeatures

X = np.arange(6).reshape(3, 2)

X

array([[0, 1],
       [2, 3],
       [4, 5]])

In [7]:
poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

X_poly = poly.fit_transform(X)

print(X_poly)

[[ 0.  1.  0.  0.  1.]
 [ 2.  3.  4.  6.  9.]
 [ 4.  5. 16. 20. 25.]]


### Discussion

Polynomial features generate additional variables based on combinations of existing features.

These new features can improve model performance when relationships between variables are non-linear.

# Polynomial Features for Regression

Polynomial features are often used to improve the performance of linear models when the relationship between variables is not strictly linear.

By adding squared terms and interaction terms, the model can capture more complex patterns.

In [11]:
from sklearn.linear_model import LinearRegression
from mglearn.datasets import make_wave

X, y = make_wave(
    n_samples=100
)

poly = PolynomialFeatures(
    degree=3,
    include_bias=False
)

X_poly = poly.fit_transform(X)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X_poly,
    y,
    random_state=42
)

model = LinearRegression()

model.fit(X_train, y_train)

print("Training score: {:.3f}".format(
    model.score(X_train, y_train)
))

print("Test score: {:.3f}".format(
    model.score(X_test, y_test)
))

Training score: 0.619
Test score: 0.685


### Discussion

Polynomial features allow linear models to learn more complex relationships by introducing higher-order terms.

This technique can improve predictive performance, but excessive polynomial degrees may lead to overfitting.

# Feature Selection

Feature selection aims to identify the most informative variables within a dataset.

Reducing the number of features can improve model interpretability, reduce computational cost, and sometimes improve generalization performance.

# Univariate Feature Selection

Univariate feature selection evaluates each feature individually and selects those that have the strongest relationship with the target variable.

In [13]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data,
    cancer.target,
    random_state=0
)

In [14]:
from sklearn.feature_selection import SelectPercentile
from sklearn.feature_selection import f_classif

select = SelectPercentile(
    f_classif,
    percentile=50
)

select.fit(X_train, y_train)

X_train_selected = select.transform(X_train)

print("X_train shape:", X_train.shape)
print("X_train_selected shape:", X_train_selected.shape)

X_train shape: (426, 30)
X_train_selected shape: (426, 15)


### Discussion

The feature selection process removes less informative variables and retains only the most relevant features.

This can simplify machine learning models and reduce noise in the dataset.

# Model-Based Feature Selection

Model-based feature selection uses a machine learning model to estimate feature importance.

Features with higher importance scores are retained, while less useful features can be removed.

In [15]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

select = SelectFromModel(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    threshold="median"
)

In [16]:
select.fit(
    X_train,
    y_train
)

X_train_selected = select.transform(
    X_train
)

print("Selected feature shape:",
      X_train_selected.shape)

Selected feature shape: (426, 15)


### Discussion

Random Forests can estimate feature importance during training.

Using these importance values, less useful features can be removed while preserving the most valuable information.

# Comparing Feature Engineering Techniques

Different feature engineering techniques address different challenges.

- One-Hot Encoding handles categorical variables.
- Binning transforms continuous variables into intervals.
- Polynomial Features create additional feature combinations.
- Feature Selection removes less informative variables.

Selecting appropriate features is often as important as choosing the machine learning algorithm itself.

# Key Takeaways

This chapter introduced several important feature engineering techniques:

- One-Hot Encoding
- Binning and Discretization
- Polynomial Features
- Feature Selection
- Model-Based Feature Selection

These methods help transform raw data into more useful representations for machine learning models.

# Conclusion

Feature engineering plays a critical role in machine learning because model performance often depends on how data is represented.

This chapter demonstrated how categorical variables can be encoded, how continuous variables can be discretized, how interaction features can be generated, and how feature selection methods can reduce complexity.

Understanding these techniques helps improve model performance and creates more effective machine learning workflows.